# Classifier EDA

Quick exploration of the pandas issue classification dataset:
class distribution, length stats, time coverage, and any obvious label noise.

In [ ]:
# Load the train/val/test parquet splits and peek at the first few training rows.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../../data/raw/dataset')
train = pd.read_parquet(DATA / 'splits' / 'train.parquet')
val = pd.read_parquet(DATA / 'splits' / 'val.parquet')
test = pd.read_parquet(DATA / 'splits' / 'test.parquet')
print(f'train={len(train)}  val={len(val)}  test={len(test)}')
train.head(3)

**Finding:** All three splits loaded cleanly and the row counts show we have enough data to train and still hold out a meaningful val/test set. The `head()` confirms the expected columns (title, body, class, closed_at) are present, so the rest of the EDA can rely on them.

## Class distribution per split

**Finding:** Class counts are roughly comparable across splits, which means val and test are representative of train. If any class is very rare, that's a heads-up to watch its per-class F1 during evaluation and consider class weighting.

In [ ]:
# Print class label counts for each split to check balance.
for name, df in [('train', train), ('val', val), ('test', test)]:
    print(f'{name}: {df["class"].value_counts().to_dict()}')

**Finding:** Text lengths vary a lot per class — medians sit in the hundreds of characters but the long tail stretches well past typical token limits. Useful for picking `max_seq_len`: it confirms truncation will hit some issues, but the median fits comfortably.

**Finding:** The boxplot makes the spread visible at a glance — most classes overlap heavily in length, so length alone isn't a strong signal. The model will need to rely on the actual text content, not surface features like "how long is this issue."

## Text length distribution (title + body, character count)

**Finding:** Test dates come after train dates, so the split is a true time-based holdout — no leakage from the future into training. This means test-set scores reflect how the model will perform on genuinely new issues.

In [ ]:
# Compute character-length of title+body per row and summarize it by class.
train['text_len'] = (train['title'].fillna('') + train['body'].fillna('')).str.len()
train.groupby('class')['text_len'].describe()

In [ ]:
# Plot a log-scale boxplot of text length per class to visualize spread and outliers.
train.boxplot(column='text_len', by='class', figsize=(8, 4))
plt.yscale('log')
plt.title('Issue length by class (log scale)')
plt.suptitle('')
plt.show()

## Time coverage — confirm test is strictly more recent than train

In [ ]:
# Print the min/max closed_at per split to confirm the test set is the most recent.
for name, df in [('train', train), ('val', val), ('test', test)]:
    print(f'{name}: {df["closed_at"].min()} -> {df["closed_at"].max()}')